# Pipeline End-to-End: Análise de Dados de Compliance Público

**Fluxo completo**: Ingestão de Dados → Bronze → Silver → Gold → Análise → Resultados

Este notebook executa todo o pipeline de dados de ponta a ponta, desde a coleta de dados brutos até os resultados analíticos finais.

---

## Seções do Notebook

1. **Configuração e Setup** — Ambiente, caminhos, modo de armazenamento
2. **Ingestão de Dados (Bronze)** — Coleta do IBGE, Portal Transparência, CGU
3. **Processamento de Dados (Silver)** — Normalização, limpeza, união de datasets
4. **Engenharia de Features (Gold)** — Criação de datasets prontos para análise
5. **Análise Exploratória de Dados** — Distribuições, correlações, outliers
6. **Análise Estatística** — Regressão OLS, testes de hipótese
7. **Aprendizado de Máquina** — ElasticNet, Random Forest para predição
8. **Análise de Clustering** — Segmentação K-means, visualização PCA
9. **Exportação de Resultados** — Salvamento de saídas, figuras, tabelas resumo

**Tempo Estimado de Execução**: 15-30 minutos (dependendo do modo de armazenamento e volume de dados)

**Modos de Armazenamento Suportados**:
- `local-only`: Armazena dados localmente (sem AWS necessário)
- `s3-only`: Armazena dados no S3 (requer AWS)
- `both`: Armazena em ambos os locais (redundância)

## ⚠️ Nota de Segurança

Antes de compartilhar este notebook:
1. **Limpe todas as saídas**: Kernel → Restart Kernel and Clear All Outputs
2. **Substitua valores de placeholder**: Atualize S3_BUCKET e AWS_PROFILE com seus valores reais
3. **Nunca commit credenciais**: Certifique-se de que não há chaves de API ou senhas no código

---


---

# 1. Configuração e Setup

## 1.1 Pacotes

In [ ]:
# Biblioteca padrão
import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime
import warnings

# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Estatística e ML
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score, silhouette_score

# Statsmodels para OLS com erros padrão robustos
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Configurar caminhos
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

# Importar módulos do projeto
from src.analysis.pt_br_loader import GoldDataLoaderPT
from src.processing.gold_transformer import GoldTransformer

print("✅ Todos os pacotes importados com sucesso")

## 1.2 Reprodutibilidade

In [ ]:
# Semente aleatória fixa para reprodutibilidade
SEED = 42
np.random.seed(SEED)

# Suprimir avisos não críticos
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Estilo de visualização
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(f"✅ Reprodutibilidade configurada: SEED={SEED}")

## 1.3 Configuração de Armazenamento

Escolha seu modo de armazenamento: `local-only`, `s3-only`, ou `both`

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO: Escolha seu modo de armazenamento e caminhos
# ═══════════════════════════════════════════════════════════════════════════════

# Modo de armazenamento: 'local-only', 's3-only', ou 'both'
STORAGE_MODE = 'local-only'  # Altere para 's3-only' ou 'both' conforme necessário

# Caminho de armazenamento local (para modos local-only ou both)
LOCAL_DATA_DIR = REPO_ROOT / 'data'  # Altere para seu caminho preferido

# Configuração S3 (para modos s3-only ou both)
import json as _json
_rtcfg_path = os.path.join('..', 'config', 'runtime_config.json')
if os.path.exists(_rtcfg_path):
    with open(_rtcfg_path) as _f:
        _rtcfg = _json.load(_f)
else:
    _rtcfg = {}
S3_BUCKET = os.environ.get('S3_BUCKET_NAME', _rtcfg.get('aws', {}).get('s3_bucket_name', ''))  # De variável de ambiente ou runtime_config.json
S3_PREFIX = 'bronze/'
AWS_PROFILE = None  # Ou 'seu-profile'  # Defina como None para credenciais padrão

# Criar diretórios locais se necessário
if STORAGE_MODE in ('local-only', 'both'):
    for layer in ['bronze', 'silver', 'gold']:
        (LOCAL_DATA_DIR / layer).mkdir(parents=True, exist_ok=True)
    print(f"✅ Diretórios locais criados em: {LOCAL_DATA_DIR}")

# Exportar variáveis de ambiente para scripts shell
os.environ['STORAGE_MODE'] = STORAGE_MODE
os.environ['LOCAL_DATA_DIR'] = str(LOCAL_DATA_DIR)
os.environ['S3_BUCKET'] = S3_BUCKET
if AWS_PROFILE:
    os.environ['AWS_PROFILE'] = AWS_PROFILE

print(f"📦 Modo de Armazenamento: {STORAGE_MODE}")
print(f"📁 Diretório de Dados Local: ./{LOCAL_DATA_DIR.name}")  # Caminho relativo apenas
print(f"☁️  Bucket S3: {S3_BUCKET} (prefixo: {S3_PREFIX})")

## 1.4 Configuração do Pipeline

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# OPÇÕES DO PIPELINE: Habilitar/desabilitar estágios do pipeline
# ═══════════════════════════════════════════════════════════════════════════════

# Defina como False para pular estágios (útil para re-executar partes específicas)
RUN_INGESTION = True      # Etapa 2: Ingestão de Dados (Bronze)
RUN_SILVER = True         # Etapa 3: Transformação Silver
RUN_GOLD = True           # Etapa 4: Engenharia de Features Gold
RUN_EDA = True            # Etapa 5: Análise Exploratória
RUN_STATISTICS = True     # Etapa 6: Análise Estatística
RUN_ML = True             # Etapa 7: Aprendizado de Máquina
RUN_CLUSTERING = True     # Etapa 8: Análise de Clustering
RUN_EXPORT = True         # Etapa 9: Exportação de Resultados

# Escopo de dados
STATES_OF_INTEREST = None  # None para todos os 27 estados, ou lista como ['SP', 'RJ', 'MG']
YEARS_OF_INTEREST = [2010, 2022]  # Anos do censo a incluir

print("📋 Configuração do Pipeline:")
print(f"   Ingestão: {RUN_INGESTION}")
print(f"   Silver: {RUN_SILVER}")
print(f"   Gold: {RUN_GOLD}")
print(f"   EDA: {RUN_EDA}")
print(f"   Estatística: {RUN_STATISTICS}")
print(f"   ML: {RUN_ML}")
print(f"   Clustering: {RUN_CLUSTERING}")
print(f"   Exportação: {RUN_EXPORT}")

---

# 2. Ingestão de Dados (Camada Bronze)

**Propósito**: Buscar dados brutos de APIs externas e armazenar com mínima transformação.

**Fontes de Dados**:
- API SIDRA IBGE: Dados do censo (população, renda, alfabetização, saneamento)
- Portal da Transparência: Transferências federais e sanções de compliance
- API BCB: Série IPCA para ajuste de renda real

**Saída**: Dados da camada Bronze (brutos, alinhados com a fonte)

In [ ]:
if RUN_INGESTION:
    print("═" * 80)
    print("ETAPA 2: INGESTÃO DE DADOS (Camada Bronze)")
    print("═" * 80)
    print(f"\nIniciado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Modo de Armazenamento: {STORAGE_MODE}\n")
    
    # Executar o script de ingestão Bronze
    script_path = REPO_ROOT / 'scripts' / '01_bronze_ingestion.sh'
    
    cmd = [str(script_path)]
    if STORAGE_MODE == 'local-only':
        cmd.append('--local-only')
    
    print("Executando ingestão Bronze...")
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_ROOT)
    
    if result.returncode == 0:
        print("✅ Ingestão Bronze concluída com sucesso")
    else:
        print("❌ Ingestão Bronze falhou")
        print("STDOUT:", result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
        print("STDERR:", result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    
    print(f"\nConcluído em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("⏭️  Pulando Ingestão de Dados (RUN_INGESTION=False)")

### Verificar Dados Bronze

In [ ]:
if RUN_INGESTION or True:  # Sempre verificar
    print("\n📁 Verificação de Dados Bronze:")
    
    if STORAGE_MODE in ('local-only', 'both'):
        bronze_dir = LOCAL_DATA_DIR / 'bronze'
        if bronze_dir.exists():
            for source_dir in bronze_dir.iterdir():
                if source_dir.is_dir():
                    files = list(source_dir.glob('*'))
                    print(f"   {source_dir.name}: {len(files)} arquivos")
        else:
            print(f"   ⚠️  Diretório Bronze não encontrado: {bronze_dir}")
    
    if STORAGE_MODE in ('s3-only', 'both'):
        print(f"   ☁️  Prefixo S3 bronze/: Verifique via AWS CLI: aws s3 ls s3://{S3_BUCKET}/bronze/")

---

# 3. Processamento de Dados (Camada Silver)

**Propósito**: Normalizar, limpar e unir datasets Bronze em tabelas Silver unificadas.

**Transformações**:
- Alinhamento de schema (nomes de colunas consistentes, tipos)
- Padronização de códigos de município (7 dígitos IBGE)
- Tratamento de valores ausentes e duplicatas
- União entre fontes de dados (censo + transferências + sanções)

**Saída**: Tabelas da camada Silver (normalizadas, prontas para análise)

In [ ]:
if RUN_SILVER:
    print("\n═" * 80)
    print("ETAPA 3: TRANSFORMAÇÃO SILVER")
    print("═" * 80)
    print(f"\nIniciado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    # Executar o script de transformação Silver
    script_path = REPO_ROOT / 'scripts' / '02_silver_transformation.sh'
    
    cmd = [str(script_path)]
    
    print("Executando transformação Silver...")
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_ROOT)
    
    if result.returncode == 0:
        print("✅ Transformação Silver concluída com sucesso")
    else:
        print("❌ Transformação Silver falhou")
        print("STDERR:", result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    
    print(f"\nConcluído em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("⏭️  Pulando Transformação Silver (RUN_SILVER=False)")

### Carregar Dados Silver para Pré-visualização

In [ ]:
# Pré-visualizar dados da camada Silver
silver_path = LOCAL_DATA_DIR / 'silver' if STORAGE_MODE in ('local-only', 'both') else None

if silver_path and silver_path.exists():
    silver_files = list(silver_path.glob('*.csv')) + list(silver_path.glob('*.parquet'))
    
    print(f"\n📁 Arquivos da Camada Silver ({len(silver_files)} arquivos):")
    for f in silver_files[:5]:  # Mostrar primeiros 5
        print(f"   - {f.name}")
    if len(silver_files) > 5:
        print(f"   ... e mais {len(silver_files) - 5}")
    
    # Carregar e pré-visualizar um dataset
    if silver_files:
        sample_df = pd.read_csv(silver_files[0]) if silver_files[0].suffix == '.csv' else pd.read_parquet(silver_files[0])
        print(f"\n📊 Dataset Silver de Amostra: {silver_files[0].name}")
        print(f"   Dimensão: {sample_df.shape}")
        print(f"   Colunas: {list(sample_df.columns[:5])}...")
        display(sample_df.head(3))

---

# 4. Engenharia de Features (Camada Gold)

**Propósito**: Criar datasets prontos para análise com features derivadas.

**Features Criadas**:
- Sanções por 100k habitantes (métrica normalizada)
- Renda real (ajustada pelo IPCA para BRL 2022)
- Indicadores regionais (Norte, Nordeste, etc.)
- Features para clustering (prontas para PCA)
- Tabela mestre no nível município-ano

**Saída**: Datasets da camada Gold (prontos para análise)

In [ ]:
if RUN_GOLD:
    print("\n═" * 80)
    print("ETAPA 4: ENGENHARIA DE FEATURES GOLD")
    print("═" * 80)
    print(f"\nIniciado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    # Executar o script de transformação Gold
    script_path = REPO_ROOT / 'scripts' / '03_gold_transformation.sh'
    
    cmd = [str(script_path)]
    
    print("Executando engenharia de features Gold...")
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_ROOT)
    
    if result.returncode == 0:
        print("✅ Transformação Gold concluída com sucesso")
    else:
        print("❌ Transformação Gold falhou")
        print("STDERR:", result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    
    print(f"\nConcluído em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("⏭️  Pulando Transformação Gold (RUN_GOLD=False)")

### Carregar Dados Gold para Análise

In [ ]:
print("\n📊 Carregando Datasets Gold para Análise...")

# Usar o GoldDataLoaderPT
if STORAGE_MODE == 's3-only':
    loader = GoldDataLoaderPT(
        bucket=S3_BUCKET,
        aws_profile=AWS_PROFILE if AWS_PROFILE else None,
        local_base_path=None
    )
else:
    # Para local-only ou both, prefira local para carregamento mais rápido
    loader = GoldDataLoaderPT(
        bucket=S3_BUCKET if STORAGE_MODE == 'both' else None,
        aws_profile=AWS_PROFILE if AWS_PROFILE else None,
        local_base_path=str(LOCAL_DATA_DIR / 'gold')
    )

# Carregar datasets
try:
    datasets = loader.load_all()
    
    print("\n✅ Datasets Gold carregados com sucesso:")
    for name, df in datasets.items():
        print(f"   {name}: {df.shape}")
    
    # Atribuir a variáveis para acesso fácil
    df_analysis = datasets.get('analysis_compliance')
    df_municipality = datasets.get('municipality_socioeconomic')
    df_state = datasets.get('state_summary')
    df_sanctions = datasets.get('sanctions_summary')
    df_clustering = datasets.get('consolidated_clustering')
    
except Exception as e:
    print(f"❌ Erro ao carregar datasets Gold: {e}")
    print("⚠️  Tentando carregar de caminhos alternativos...")
    
    # Fallback: tentar carregar diretamente dos caminhos
    gold_path = LOCAL_DATA_DIR / 'gold'
    if gold_path.exists():
        csv_files = list(gold_path.glob('*.csv'))
        for f in csv_files:
            var_name = f.stem.replace('-', '_').replace(' ', '_')
            globals()[f'df_{var_name}'] = pd.read_csv(f)
            print(f"   Carregado {f.name} como df_{var_name}")

---

# 5. Análise Exploratória de Dados

**Propósito**: Perfil da qualidade dos dados, inspeção de distribuições, identificação de padrões.

**Seções**:
- 5.1 Visão Geral dos Dados
- 5.2 Análise de Distribuições
- 5.3 Comparações Regionais
- 5.4 Matriz de Correlação

## 5.1 Visão Geral dos Dados

In [ ]:
if RUN_EDA and 'df_analysis' in globals() and df_analysis is not None:
    print("═" * 80)
    print("ETAPA 5: ANÁLISE EXPLORATÓRIA DE DADOS")
    print("═" * 80)
    
    print("\n📊 Visão Geral do Dataset de Análise Principal:")
    print(f"   Dimensão: {df_analysis.shape}")
    print(f"\n   Colunas:")
    for col in df_analysis.columns:
        print(f"      - {col}")
    
    print(f"\n   Valores Ausentes:")
    missing = df_analysis.isnull().sum()
    for col, count in missing[missing > 0].items():
        pct = count / len(df_analysis) * 100
        print(f"      {col}: {count} ({pct:.1f}%)")
    if missing.sum() == 0:
        print("      ✅ Sem valores ausentes")
    
    print(f"\n   Tipos de Dados:")
    print(df_analysis.dtypes)
    
    display(df_analysis.describe())
else:
    print("⏭️  Pulando AED (RUN_EDA=False ou dados não carregados)")

## 5.2 Análise de Distribuições

In [ ]:
if RUN_EDA and 'df_analysis' in globals() and df_analysis is not None:
    
    # Variáveis chave para visualização
    key_vars = ['sancoes_por_100k', 'renda_media', 'taxa_analfabetismo', 'taxa_urbanizacao']
    key_vars = [v for v in key_vars if v in df_analysis.columns]
    
    if key_vars:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes = axes.flatten()
        
        for i, var in enumerate(key_vars[:4]):
            if var in df_analysis.columns:
                axes[i].hist(df_analysis[var].dropna(), bins=30, edgecolor='black', alpha=0.7)
                axes[i].set_title(f'Distribuição: {var}')
                axes[i].set_xlabel(var)
                axes[i].set_ylabel('Frequência')
        
        plt.suptitle('Distribuições de Variáveis Principais', fontsize=14, y=1.02)
        plt.tight_layout()
        plt.show()
        
        # Salvar figura
        fig.savefig(REPO_ROOT / 'outputs' / 'eda_distribuicoes.png', dpi=300, bbox_inches='tight')
        print("\n✅ Gráficos de distribuição salvos em outputs/eda_distribuicoes.png")

## 5.3 Comparações Regionais

In [ ]:
if RUN_EDA and 'df_analysis' in globals() and df_analysis is not None:
    
    if 'regiao' in df_analysis.columns and 'sancoes_por_100k' in df_analysis.columns:
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        regional_data = df_analysis.groupby('regiao')['sancoes_por_100k'].mean().sort_values(ascending=False)
        
        regional_data.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
        ax.set_title('Média de Sanções por 100k Habitantes por Região', fontsize=12)
        ax.set_xlabel('Região')
        ax.set_ylabel('Sanções por 100k')
        ax.tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()
        
        # Salvar figura
        fig.savefig(REPO_ROOT / 'outputs' / 'sancoes_regionais.png', dpi=300, bbox_inches='tight')
        
        print("\n📊 Resumo de Sanções Regionais:")
        print(regional_data.to_frame().round(2))

## 5.4 Matriz de Correlação

In [ ]:
if RUN_EDA and 'df_analysis' in globals() and df_analysis is not None:
    
    # Selecionar colunas numéricas para correlação
    numeric_cols = df_analysis.select_dtypes(include=[np.number]).columns
    numeric_cols = [c for c in numeric_cols if not c.endswith('_id') and c != 'ano']
    
    if len(numeric_cols) > 2:
        
        # Calcular matriz de correlação
        corr_matrix = df_analysis[numeric_cols].corr()
        
        # Plotar
        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, 
                    square=True, fmt='.2f', cbar_kws={"shrink": .8}, ax=ax)
        ax.set_title('Matriz de Correlação de Variáveis Principais', fontsize=12)
        
        plt.tight_layout()
        plt.show()
        
        # Salvar figura
        fig.savefig(REPO_ROOT / 'outputs' / 'matriz_correlacao.png', dpi=300, bbox_inches='tight')
        
        # Imprimir correlações mais fortes com sanções
        if 'sancoes_por_100k' in corr_matrix.columns:
            sanctions_corr = corr_matrix['sancoes_por_100k'].drop('sancoes_por_100k').abs().sort_values(ascending=False)
            print("\n📊 Correlações Mais Fortes com Sanções por 100k:")
            for var, corr in sanctions_corr.head(5).items():
                direction = "positiva" if corr_matrix['sancoes_por_100k'][var] > 0 else "negativa"
                print(f"   {var}: r = {corr_matrix['sancoes_por_100k'][var]:.3f} ({direction})")

---

# 6. Análise Estatística

**Propósito**: Modelagem de regressão OLS com erros padrão robustos.

**Modelos**:
- Modelo 1: Apenas renda (baseline)
- Modelo 2: Renda + controles (alfabetização, urbanização)
- Modelo 3: Modelo completo com efeitos regionais

In [ ]:
if RUN_STATISTICS and 'df_analysis' in globals() and df_analysis is not None:
    
    print("\n═" * 80)
    print("ETAPA 6: ANÁLISE ESTATÍSTICA (Regressão OLS)")
    print("═" * 80)
    
    # Preparar dados
    model_df = df_analysis.copy()
    
    # Definir variáveis
    dependent = 'sancoes_por_100k'
    independent = [
        'renda_media',
        'taxa_analfabetismo',
        'taxa_urbanizacao',
        'indice_gini'
    ]
    
    # Filtrar para colunas disponíveis
    available_vars = [v for v in [dependent] + independent if v in model_df.columns]
    
    if len(available_vars) < 2:
        print("⚠️  Variáveis insuficientes para regressão. Colunas disponíveis:")
        print(list(model_df.columns))
    else:
        # Criar casos completos
        regression_df = model_df[available_vars].dropna()
        
        print(f"\n📊 Dataset de Regressão: {regression_df.shape[0]} observações, {len(available_vars)} variáveis")
        
        # Modelo 1: Apenas renda
        if 'renda_media' in regression_df.columns:
            print("\n📈 Modelo 1: Apenas Renda")
            X1 = sm.add_constant(regression_df['renda_media'])
            y = regression_df[dependent]
            model1 = sm.OLS(y, X1).fit(cov_type='HC3')  # SE robustos
            print(f"   R² = {model1.rsquared:.3f}")
            print(f"   Coeficiente renda: {model1.params['renda_media']:.4f} (p={model1.pvalues['renda_media']:.4f})")
        
        # Modelo 2: Especificação completa
        independent_available = [v for v in independent if v in regression_df.columns]
        if len(independent_available) >= 2:
            print("\n📈 Modelo 2: Especificação Completa")
            X2 = sm.add_constant(regression_df[independent_available])
            model2 = sm.OLS(y, X2).fit(cov_type='HC3')
            print(f"   R² = {model2.rsquared:.3f}")
            print(f"   R² Ajustado = {model2.rsquared_adj:.3f}")
            print("\n   Coeficientes:")
            for var in independent_available:
                coef = model2.params[var]
                pval = model2.pvalues[var]
                stars = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
                print(f"      {var}: {coef:.4f} (p={pval:.4f}){stars}")
        
        # Armazenar resultados para resumo
        regression_results = {
            'model1_r2': model1.rsquared if 'model1' in locals() else None,
            'model2_r2': model2.rsquared if 'model2' in locals() else None,
            'income_coef': model2.params['renda_media'] if 'model2' in locals() and 'renda_media' in model2.params else None
        }
else:
    print("⏭️  Pulando Análise Estatística (RUN_STATISTICS=False ou dados não carregados)")

---

# 7. Aprendizado de Máquina

**Propósito**: Modelos supervisionados para predição de sanções.

**Modelos**:
- ElasticNet (linear regularizado)
- Random Forest (ensemble, não-linear)

In [ ]:
if RUN_ML and 'df_analysis' in globals() and df_analysis is not None:
    
    print("\n═" * 80)
    print("ETAPA 7: APRENDIZADO DE MÁQUINA")
    print("═" * 80)
    
    # Preparar dataset ML
    ml_vars = ['sancoes_por_100k', 'renda_media', 'taxa_analfabetismo', 'taxa_urbanizacao']
    ml_vars = [v for v in ml_vars if v in df_analysis.columns]
    
    if len(ml_vars) < 2:
        print("⚠️  Variáveis insuficientes para ML")
    else:
        ml_df = df_analysis[ml_vars].dropna()
        
        X = ml_df.drop('sancoes_por_100k', axis=1)
        y = ml_df['sancoes_por_100k']
        
        # Divisão treino/teste
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=SEED
        )
        
        print(f"\n📊 Dataset ML: {len(X_train)} treino, {len(X_test)} teste amostras")
        print(f"   Features: {list(X.columns)}")
        
        # Escalar features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Modelo 1: ElasticNet
        print("\n📈 Modelo 1: ElasticNet")
        enet = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=SEED)
        enet.fit(X_train_scaled, y_train)
        enet_pred = enet.predict(X_test_scaled)
        enet_r2 = r2_score(y_test, enet_pred)
        enet_rmse = np.sqrt(mean_squared_error(y_test, enet_pred))
        
        print(f"   R² (teste): {enet_r2:.3f}")
        print(f"   RMSE: {enet_rmse:.2f}")
        print(f"   Coeficientes não-zero: {np.sum(enet.coef_ != 0)}/{len(enet.coef_)}")
        
        # Modelo 2: Random Forest
        print("\n📈 Modelo 2: Random Forest")
        rf = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            random_state=SEED,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)  # RF não requer escalonamento
        rf_pred = rf.predict(X_test)
        rf_r2 = r2_score(y_test, rf_pred)
        rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
        
        print(f"   R² (teste): {rf_r2:.3f}")
        print(f"   RMSE: {rf_rmse:.2f}")
        
        # Importância de features
        print("\n📊 Importância de Features (Random Forest):")
        importance = pd.Series(rf.feature_importances_, index=X.columns)
        importance = importance.sort_values(ascending=False)
        for feat, imp in importance.items():
            print(f"   {feat}: {imp:.3f}")
        
        # Gráfico de comparação de modelos
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Previsões ElasticNet
        axes[0].scatter(y_test, enet_pred, alpha=0.6, edgecolors='black', linewidth=0.5)
        axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        axes[0].set_xlabel('Real')
        axes[0].set_ylabel('Predito')
        axes[0].set_title(f'ElasticNet (R² = {enet_r2:.3f})')
        
        # Previsões Random Forest
        axes[1].scatter(y_test, rf_pred, alpha=0.6, edgecolors='black', linewidth=0.5, color='green')
        axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        axes[1].set_xlabel('Real')
        axes[1].set_ylabel('Predito')
        axes[1].set_title(f'Random Forest (R² = {rf_r2:.3f})')
        
        plt.suptitle('Performance de Modelos de Machine Learning', fontsize=14)
        plt.tight_layout()
        plt.show()
        
        # Salvar figura
        fig.savefig(REPO_ROOT / 'outputs' / 'ml_predicoes.png', dpi=300, bbox_inches='tight')
        
        # Armazenar resultados
        ml_results = {
            'elasticnet_r2': enet_r2,
            'elasticnet_rmse': enet_rmse,
            'rf_r2': rf_r2,
            'rf_rmse': rf_rmse,
            'best_model': 'Random Forest' if rf_r2 > enet_r2 else 'ElasticNet'
        }
else:
    print("⏭️  Pulando Aprendizado de Máquina (RUN_ML=False ou dados não carregados)")

---

# 8. Análise de Clustering

**Propósito**: Segmentação não-supervisionada de municípios.

**Método**: Clustering K-means com visualização PCA

In [ ]:
if RUN_CLUSTERING and 'df_clustering' in globals() and df_clustering is not None:
    
    print("\n═" * 80)
    print("ETAPA 8: ANÁLISE DE CLUSTERING")
    print("═" * 80)
    
    # Preparar dados de clustering
    cluster_vars = ['renda_media', 'taxa_analfabetismo', 'taxa_urbanizacao', 'sancoes_por_100k']
    cluster_vars = [v for v in cluster_vars if v in df_clustering.columns]
    
    if len(cluster_vars) < 2:
        print("⚠️  Variáveis insuficientes para clustering")
    else:
        cluster_df = df_clustering[cluster_vars].dropna()
        
        print(f"\n📊 Dataset de Clustering: {cluster_df.shape[0]} municípios, {len(cluster_vars)} features")
        
        # Escalar features
        scaler = StandardScaler()
        cluster_scaled = scaler.fit_transform(cluster_df)
        
        # Determinar K ótimo (método do cotovelo simplificado)
        K = 4  # Baseado em análise anterior: 2 bulk + 2 outliers
        
        # Ajustar K-means
        print(f"\n📈 Clustering K-Means (K={K})")
        kmeans = KMeans(n_clusters=K, random_state=SEED, n_init=10)
        cluster_labels = kmeans.fit_predict(cluster_scaled)
        
        # Score de silhueta
        sil_score = silhouette_score(cluster_scaled, cluster_labels)
        print(f"   Score de Silhueta: {sil_score:.3f}")
        
        # Tamanhos dos clusters
        cluster_sizes = pd.Series(cluster_labels).value_counts().sort_index()
        print(f"\n   Tamanhos dos Clusters:")
        for i, size in cluster_sizes.items():
            print(f"      Cluster {i}: {size} municípios")
        
        # Características dos clusters
        cluster_df['cluster'] = cluster_labels
        print(f"\n📊 Características dos Clusters (valores médios):")
        cluster_summary = cluster_df.groupby('cluster')[cluster_vars].mean()
        print(cluster_summary.round(2))
        
        # PCA para visualização
        print("\n📈 Visualização PCA")
        pca = PCA(n_components=2)
        pca_result = pca.fit_transform(cluster_scaled)
        
        print(f"   Variância Explicada: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")
        print(f"   Total: {sum(pca.explained_variance_ratio_):.1%}")
        
        # Plotar
        fig, ax = plt.subplots(figsize=(10, 8))
        
        colors = ['blue', 'green', 'orange', 'red']
        for i in range(K):
            mask = cluster_labels == i
            ax.scatter(
                pca_result[mask, 0],
                pca_result[mask, 1],
                c=colors[i],
                label=f'Cluster {i} (n={cluster_sizes[i]})',
                alpha=0.6,
                edgecolors='black',
                linewidth=0.5
            )
        
        ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variância)')
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variância)')
        ax.set_title('Clusters de Municípios (K-Means + PCA)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Salvar figura
        fig.savefig(REPO_ROOT / 'outputs' / 'clustering_pca.png', dpi=300, bbox_inches='tight')
        
        # Armazenar resultados
        clustering_results = {
            'k': K,
            'silhouette_score': sil_score,
            'cluster_sizes': cluster_sizes.to_dict(),
            'pca_variance': sum(pca.explained_variance_ratio_)
        }
else:
    print("⏭️  Pulando Clustering (RUN_CLUSTERING=False ou dados não carregados)")

---

# 9. Exportação de Resultados

**Propósito**: Salvar saídas, figuras e tabelas resumo para integração na tese.

In [ ]:
if RUN_EXPORT:
    
    print("\n═" * 80)
    print("ETAPA 9: EXPORTAÇÃO DE RESULTADOS")
    print("═" * 80)
    
    # Criar diretório de saídas
    outputs_dir = REPO_ROOT / 'outputs'
    outputs_dir.mkdir(exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Exportação 1: Estatísticas resumo
    summary_data = {
        'run_timestamp': timestamp,
        'storage_mode': STORAGE_MODE,
        'data_sources': ['IBGE', 'Portal da Transparência', 'Sanções CGU'],
    }
    
    # Adicionar resultados de regressão se disponíveis
    if 'regression_results' in globals():
        summary_data['regression'] = regression_results
    
    # Adicionar resultados de ML se disponíveis
    if 'ml_results' in globals():
        summary_data['machine_learning'] = ml_results
    
    # Adicionar resultados de clustering se disponíveis
    if 'clustering_results' in globals():
        summary_data['clustering'] = clustering_results
    
    # Salvar resumo JSON
    summary_path = outputs_dir / f'resumo_analise_{timestamp}.json'
    with open(summary_path, 'w') as f:
        json.dump(summary_data, f, indent=2, default=str)
    print(f"\n✅ Resumo salvo: {summary_path}")
    
    # Exportação 2: Tabelas de dados (se dados de análise disponíveis)
    if 'df_analysis' in globals() and df_analysis is not None:
        # Resumo estadual
        if 'state_code' in df_analysis.columns:
            state_summary = df_analysis.groupby('state_code').agg({
                'sancoes_por_100k': 'mean',
                'renda_media': 'mean'
            }).round(2)
            state_summary.to_csv(outputs_dir / 'resumo_estadual.csv')
            print("✅ Resumo estadual exportado")
    
    # Exportação 3: Lista de figuras principais
    figures = list(outputs_dir.glob('*.png'))
    print(f"\n📊 Figuras Geradas ({len(figures)}):")
    for fig in sorted(figures):
        print(f"   - {fig.name}")
    
    print(f"\n✅ Todas as saídas salvas em: {outputs_dir}")
    
    # Resumo final
    print("\n" + "═" * 80)
    print("PIPELINE CONCLUÍDO")
    print("═" * 80)
    print(f"\nTimestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Modo de Armazenamento: {STORAGE_MODE}")
    print(f"Diretório de Saída: {outputs_dir}")
    
else:
    print("⏭️  Pulando Exportação de Resultados (RUN_EXPORT=False)")

---

# Apêndice: Referência Rápida

## Executando Seções Específicas

Para re-executar apenas partes específicas, altere a configuração na Seção 1.4:

```python
# Exemplo: Executar apenas análise (pular ingestão/processamento)
RUN_INGESTION = False
RUN_SILVER = False
RUN_GOLD = False
RUN_EDA = True
RUN_STATISTICS = True
RUN_ML = True
RUN_CLUSTERING = True
```

## Troca Rápida de Modo de Armazenamento

```python
# Apenas local (sem AWS)
STORAGE_MODE = 'local-only'
LOCAL_DATA_DIR = Path('/caminho/para/dados')

# Apenas S3 (requer AWS)
STORAGE_MODE = 's3-only'
S3_BUCKET = 'seu-bucket'

# Ambos (redundância)
STORAGE_MODE = 'both'
```

## Tempo de Execução Esperado

| Estágio | Local | S3 | Ambos |
|---------|-------|-----|------|
| Ingestão | 5-10 min | 5-10 min | 10-15 min |
| Silver | 2-3 min | 2-3 min | 3-5 min |
| Gold | 1-2 min | 1-2 min | 2-3 min |
| Análise | 2-3 min | 2-3 min | 2-3 min |
| **Total** | **10-18 min** | **10-18 min** | **17-26 min** |

## Arquivos de Saída

O pipeline gera:
- `outputs/eda_distribuicoes.png` — Distribuições de variáveis
- `outputs/sancoes_regionais.png` — Comparação regional
- `outputs/matriz_correlacao.png` — Mapa de calor de correlações
- `outputs/ml_predicoes.png` — Performance dos modelos ML
- `outputs/clustering_pca.png` — Visualização dos clusters
- `outputs/resumo_analise_YYYYMMDD_HHMMSS.json` — Resumo dos resultados
- `outputs/resumo_estadual.csv` — Estatísticas por estado

---

## Solução de Problemas

### Arquivos de dados ausentes
```python
# Re-executar estágios específicos do pipeline
RUN_INGESTION = True
RUN_SILVER = True
RUN_GOLD = True
```

### Erros de autenticação AWS
```bash
# Verificar credenciais
aws sts get-caller-identity

# Ou mudar para modo local-only
STORAGE_MODE = 'local-only'
```

### Problemas de memória com datasets grandes
```python
# Filtrar para estados específicos
STATES_OF_INTEREST = ['SP', 'RJ', 'MG']
```